# Auditoría de datos, construcción del target e ingeniería de variables

## Objetivo

En este notebook preparo los datos para un modelo que busca predecir si un pedido de Olist llegará con demora.

El trabajo sigue este orden:

1. cargar los datos originales;
2. revisar su estructura y calidad;
3. comprobar que cada fila represente un pedido;
4. aplicar el target oficial del equipo;
5. construir variables conocidas al momento de aprobar el pago;
6. controlar que no exista fuga de información;
7. revisar valores faltantes, imposibles y extremos.

La lógica reutilizable se encuentra en `src/`. El notebook sirve para comprobarla y explicarla, sin volver a programarla por separado.


## 1. Preparación del entorno e importaciones


In [1]:
from pathlib import Path
import sys

# El notebook puede abrirse desde la carpeta notebooks o desde la raíz.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data_loading import load_raw_tables
from src.features import build_features, feature_columns, leaky_columns
from src.target import TARGET_COL, build_target


## 2. Carga de las tablas originales

Uso la función compartida `load_raw_tables()` en lugar de una ruta personal de Windows. De esta forma, cualquier integrante puede descargar los CSV en `data/raw/` y ejecutar el mismo notebook.


In [2]:
tables = load_raw_tables()
db = tables["orders"].copy()

formas_tablas = {
    nombre: tabla.shape
    for nombre, tabla in tables.items()
}

display(formas_tablas)


{'orders': (99441, 8),
 'items': (112650, 7),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'customers': (99441, 5),
 'payments': (103886, 5)}

## 3. Exploración inicial de pedidos

Primero observo ejemplos y la forma general de la tabla. `head()` muestra las primeras filas y `sample()` permite mirar registros ubicados en otras posiciones.


In [3]:
print("Primeras cinco filas")
display(db.head())

print("Muestra aleatoria reproducible")
display(db.sample(5, random_state=42))

print("Dimensiones:", db.shape)


Primeras cinco filas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


Muestra aleatoria reproducible


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
52263,b9a6c5f5df52c7226ac85aee7524c27f,f160aaf480efdfa7268f0fa535f73e76,delivered,2018-06-12 20:07:44,2018-06-12 20:44:26,2018-06-13 13:09:00,2018-06-19 12:44:08,2018-07-17
46645,261e71d2349c713eafa9f3df5972b95d,d6708bbbd2d419475869a84e41f620a1,delivered,2018-01-20 12:15:57,2018-01-20 12:37:13,2018-01-25 21:42:52,2018-01-30 11:32:35,2018-02-15
37546,67b50899f52995848c427e361e10dde3,1b353c00c71689afba44554e43cc5a76,delivered,2018-06-16 21:24:10,2018-06-16 21:36:59,2018-06-21 13:55:00,2018-06-27 13:17:27,2018-07-16
94756,32733fc014b67ef70fa6039dd8c6ba82,ad66699beab381d0747605f108cc02b4,delivered,2017-08-30 21:12:28,2017-08-31 02:50:24,2017-09-12 20:16:46,2017-09-25 17:53:23,2017-09-22
14771,39a70e9e9b729b11dee34ac12478597f,20c678bfce3c8252288f70dd2bee5e51,delivered,2017-08-10 21:26:25,2017-08-10 21:44:13,2017-08-11 19:11:16,2017-08-22 16:45:00,2017-09-12


Dimensiones: (99441, 8)


In [4]:
print("Información general")
db.info()

print("Tipos de datos")
display(db.dtypes)

print("Valores faltantes")
display(db.isna().sum())

print("Filas completamente duplicadas:", db.duplicated().sum())


Información general
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB
Tipos de datos


order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Valores faltantes


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Filas completamente duplicadas: 0


### Granularidad

La granularidad indica qué representa una fila. Si la cantidad de filas coincide con la cantidad de `order_id` únicos, la tabla tiene una fila por pedido y no necesita agruparse para este análisis.


In [5]:
cantidad_filas = len(db)
pedidos_unicos = db["order_id"].nunique()

print("Cantidad de filas:", cantidad_filas)
print("Pedidos únicos:", pedidos_unicos)
print("¿Hay una fila por pedido?:", cantidad_filas == pedidos_unicos)
print("order_id repetidos:", db["order_id"].duplicated().sum())


Cantidad de filas: 99441
Pedidos únicos: 99441
¿Hay una fila por pedido?: True
order_id repetidos: 0


### Fechas y estados

Las fechas se convierten al cargar la tabla. En esta parte verifico sus tipos y cuántos valores faltan. También reviso los estados, porque el target solo puede observarse en pedidos entregados.


In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

display(db[date_columns].dtypes.rename("tipo"))
display(db[date_columns].isna().sum().rename("cantidad_nulos"))
display(db["order_status"].value_counts(dropna=False))


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
Name: tipo, dtype: object

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
Name: cantidad_nulos, dtype: int64

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 4. Construcción del target oficial

El target se llama `pedido_retrasado`:

- `1`: el pedido llegó en un día posterior al prometido;
- `0`: llegó el día prometido o antes.

Utilizo `build_target()` para aplicar la misma definición que el resto del equipo. La función conserva entregas observables y limita el estudio a compras desde enero de 2017 hasta agosto de 2018.


In [7]:
db_modelo = build_target(db)

conteo_target = db_modelo[TARGET_COL].value_counts().sort_index()
porcentaje_target = (
    db_modelo[TARGET_COL]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Dimensiones:", db_modelo.shape)
display(pd.DataFrame({"cantidad": conteo_target, "porcentaje": porcentaje_target}))


Dimensiones: (96203, 9)


,cantidad,porcentaje
pedido_retrasado,,
0,89672,93.21
1,6531,6.79


In [8]:
print("Cantidad de filas:", len(db_modelo))
print("Pedidos únicos:", db_modelo["order_id"].nunique())
print("¿Hay una fila por pedido?:", db_modelo["order_id"].is_unique)


Cantidad de filas: 96203
Pedidos únicos: 96203
¿Hay una fila por pedido?: True


### Por qué antes aparecían 6,77 % y 8,11 %

La fecha estimada viene a las 00:00, pero la entrega real incluye una hora. Si se comparan los timestamps completos, una entrega realizada durante el día prometido puede quedar marcada como tardía.

Para que la comparación sea justa, aplico las dos reglas sobre la misma población histórica. Estas cifras sirven como auditoría; el modelo utiliza el target oficial del período de estudio.


In [9]:
target_dia_historico = build_target(
    db,
    by_calendar_day=True,
    only_study_period=False,
)[["order_id", TARGET_COL]].rename(columns={TARGET_COL: "regla_por_dia"})

target_hora_historico = build_target(
    db,
    by_calendar_day=False,
    only_study_period=False,
)[["order_id", TARGET_COL]].rename(columns={TARGET_COL: "regla_por_fecha_y_hora"})

comparacion_target = target_dia_historico.merge(
    target_hora_historico,
    on="order_id",
    validate="one_to_one",
)

print(f"Tasa histórica por día: {comparacion_target['regla_por_dia'].mean():.2%}")
print(f"Tasa histórica con fecha y hora: {comparacion_target['regla_por_fecha_y_hora'].mean():.2%}")

display(pd.crosstab(
    comparacion_target["regla_por_dia"],
    comparacion_target["regla_por_fecha_y_hora"],
    rownames=["Regla por día"],
    colnames=["Regla por fecha y hora"],
))


Tasa histórica por día: 6.77%
Tasa histórica con fecha y hora: 8.11%


Regla por fecha y hora,0,1
Regla por día,,
0,88644,1292
1,0,6534


**Interpretación:** 6,77 % y 8,11 % correspondían a todos los pedidos entregados observables, pero con reglas diferentes. El 6,79 % es el resultado por día calendario después de aplicar también el período oficial. Por eso, la cifra utilizada por el proyecto es 6,79 %.


## 5. Construcción de variables predictoras

La predicción se realiza al aprobarse el pago (`anchor="aprobacion"`). La pregunta es: con lo que sabemos en ese momento, ¿el pedido tiene riesgo de llegar tarde?

`build_features()` une las tablas y agrega la información a nivel pedido. Los registros sin fecha de aprobación se excluyen porque no conocemos el momento elegido para predecir.


In [10]:
tables_modelo = {**tables, "orders": db_modelo}

df_features = build_features(
    **tables_modelo,
    anchor="aprobacion",
)

print("Pedidos antes de crear variables:", len(db_modelo))
print("Pedidos después de crear variables:", len(df_features))
print("Pedidos excluidos por no tener aprobación:", len(db_modelo) - len(df_features))
display(df_features.head())


Pedidos antes de crear variables: 96203
Pedidos después de crear variables: 96189
Pedidos excluidos por no tener aprobación: 14


,order_id,order_purchase_timestamp,prediction_time,purchase_month,purchase_dayofweek,purchase_hour,promised_days,approval_hours,pedido_retrasado,customer_state,n_items,n_sellers,total_price,total_freight,total_weight_g,total_volume_cm3,main_payment_type,max_installments,freight_ratio,seller_other_state
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-02 11:07:15,10,0,10,16,0.178333,0,SP,1,1,29.99,8.72,500.0,1976.0,voucher,1.0,0.290764,False
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07-26 03:24:27,7,1,20,20,30.713889,0,BA,1,1,118.70,22.76,400.0,4693.0,boleto,1.0,0.191744,True
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-08 08:55:23,8,2,8,27,0.276111,0,GO,1,1,159.90,19.22,420.0,9576.0,credit_card,3.0,0.120200,True
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-11-18 19:45:59,11,5,19,27,0.298056,0,RN,1,1,45.00,27.20,450.0,6000.0,credit_card,1.0,0.604444,True
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-13 22:20:29,2,1,21,13,1.030556,0,SP,1,1,19.90,8.72,250.0,11475.0,credit_card,1.0,0.438191,False


### Variables utilizadas

Las variables incluyen momento de compra, plazo prometido, carrito, pago, producto y ubicación. `prediction_time` y `order_id` ayudan a ordenar e identificar, pero no forman parte de las variables de entrada del modelo.


In [11]:
variables_oficiales = feature_columns(anchor="aprobacion")

print("Cantidad de variables predictoras:", len(variables_oficiales))
display(pd.Series(variables_oficiales, name="variable"))

print("¿Sigue existiendo una fila por pedido?:", df_features["order_id"].is_unique)
print("order_id repetidos:", df_features["order_id"].duplicated().sum())


Cantidad de variables predictoras: 16


0         purchase_month
1     purchase_dayofweek
2          purchase_hour
3          promised_days
4                n_items
5              n_sellers
6            total_price
7          total_freight
8          freight_ratio
9         total_weight_g
10      total_volume_cm3
11      max_installments
12        approval_hours
13        customer_state
14     main_payment_type
15    seller_other_state
Name: variable, dtype: str

¿Sigue existiendo una fila por pedido?: True
order_id repetidos: 0


## 6. Control anti-leakage

Una fuga ocurre cuando el modelo recibe información que todavía no existiría al momento de predecir. Por ejemplo, no puede usar la fecha real de entrega para anticipar si el pedido se retrasará.

La fecha real se necesita para crear el target histórico, pero no se incluye entre los predictores.


In [12]:
columnas_prohibidas = leaky_columns(anchor="aprobacion")
fugas_encontradas = sorted(set(variables_oficiales).intersection(columnas_prohibidas))

print("Columnas prohibidas:")
display(pd.Series(columnas_prohibidas, name="columna"))

print("Variables con fuga encontradas:", fugas_encontradas)
assert not fugas_encontradas, "Se encontraron variables con fuga"


Columnas prohibidas:


0     order_delivered_carrier_date
1    order_delivered_customer_date
2                     order_status
3              shipping_limit_date
4                     review_score
Name: columna, dtype: str

Variables con fuga encontradas: []


## 7. Calidad de las variables construidas

Primero reviso nulos e infinitos. Los faltantes no se rellenan manualmente aquí: el pipeline aprende la mediana usando solamente entrenamiento, para no mezclar información de validación o test.


In [13]:
resumen_nulos = pd.DataFrame({
    "cantidad_nulos": df_features[variables_oficiales].isna().sum(),
    "porcentaje_nulos": (
        df_features[variables_oficiales].isna().mean().mul(100).round(3)
    ),
}).sort_values("cantidad_nulos", ascending=False)

display(resumen_nulos)

cantidad_infinitos = np.isinf(
    df_features.select_dtypes(include="number")
).sum()

print("Columnas con infinitos:")
display(cantidad_infinitos[cantidad_infinitos > 0])


,cantidad_nulos,porcentaje_nulos
total_weight_g,22,0.023
total_volume_cm3,16,0.017
max_installments,2,0.002
purchase_month,0,0.000
n_items,0,0.000
purchase_dayofweek,0,0.000
purchase_hour,0,0.000
promised_days,0,0.000
total_freight,0,0.000
total_price,0,0.000


Columnas con infinitos:


Series([], dtype: int64)

### Hallazgo: medidas físicas y cuotas inválidas

En la primera revisión aparecieron pesos, volúmenes y cuotas iguales a cero. Al volver a las tablas originales se comprobó que había productos con medidas faltantes, productos con peso cero y pagos con cero cuotas.

Se corrigió `src/features.py` para mantener esos casos como faltantes. De esta manera el modelo no interpreta que un producto realmente pesa cero y el pipeline puede imputarlos más adelante.


In [14]:
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

productos_con_faltantes = tables["products"][physical_columns].isna().any(axis=1)
productos_con_no_positivos = tables["products"][physical_columns].le(0).any(axis=1)
pagos_con_cuotas_no_positivas = tables["payments"]["payment_installments"].le(0)

print("Productos con alguna medida faltante:", productos_con_faltantes.sum())
print("Productos con alguna medida <= 0:", productos_con_no_positivos.sum())
print("Pagos con cuotas <= 0:", pagos_con_cuotas_no_positivas.sum())

display(tables["products"].loc[
    productos_con_faltantes | productos_con_no_positivos,
    ["product_id", *physical_columns],
])

display(tables["payments"].loc[pagos_con_cuotas_no_positivas])


Productos con alguna medida faltante: 2
Productos con alguna medida <= 0: 4
Pagos con cuotas <= 0: 2


,product_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,NaN,NaN,NaN,NaN
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,0.0,30.0,25.0,30.0
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN
32079,e673e90efa65a5409ff4196c038bb5af,0.0,30.0,25.0,30.0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


## 8. Exploración de algunas variables

Esta parte no modifica los datos. Busca entender sus rangos y su relación inicial con el target. Los valores extremos no se eliminan automáticamente: primero hay que distinguir entre errores y casos raros pero posibles.


In [15]:
numeric_review = [
    "promised_days",
    "approval_hours",
    "n_items",
    "n_sellers",
    "total_price",
    "total_freight",
    "freight_ratio",
    "total_weight_g",
    "total_volume_cm3",
    "max_installments",
]

display(df_features[numeric_review].describe().T)


,count,mean,std,min,25%,50%,75%,max
promised_days,96189.0,24.283151,8.600569,3.00,19.000000,24.000000,29.000000,1.560000e+02
approval_hours,96189.0,10.248453,20.520523,0.00,0.215000,0.342222,14.423889,7.414436e+02
n_items,96189.0,1.142095,0.538491,1.00,1.000000,1.000000,1.000000,2.100000e+01
n_sellers,96189.0,1.013910,0.123597,1.00,1.000000,1.000000,1.000000,5.000000e+00
total_price,96189.0,137.002230,209.132474,0.85,45.900000,86.500000,149.900000,1.344000e+04
total_freight,96189.0,22.785131,21.567445,0.00,13.850000,17.170000,24.010000,1.794960e+03
freight_ratio,96189.0,0.308437,0.311754,0.00,0.132028,0.224374,0.380585,2.144706e+01
total_weight_g,96167.0,2386.413083,4771.702099,2.00,300.000000,750.000000,2050.000000,1.844000e+05
total_volume_cm3,96173.0,17322.980348,30153.555810,168.00,2964.000000,7225.000000,19840.000000,1.476000e+06
max_installments,96187.0,2.926404,2.711500,1.00,1.000000,2.000000,4.000000,2.400000e+01


In [16]:
print("Pedidos con plazo prometido menor o igual a cero:",
      (df_features["promised_days"] <= 0).sum())

display(
    df_features.nlargest(10, "promised_days")[[
        "order_id",
        "order_purchase_timestamp",
        "promised_days",
        "total_price",
        "total_freight",
        "customer_state",
        TARGET_COL,
    ]]
)


Pedidos con plazo prometido menor o igual a cero: 0


,order_id,order_purchase_timestamp,promised_days,total_price,total_freight,customer_state,pedido_retrasado
55297,eec7f369423b033e549c02f3c5381205,2018-02-06 20:44:56,156,252.99,19.02,RS,0
38786,0607f0efea4b566f1eb8f7d3c2397320,2018-03-06 09:47:07,150,125.98,20.18,SP,0
15298,c72727d29cde4cf870d569bf65edabfd,2017-02-07 18:01:15,147,25.90,26.09,MA,0
83609,c2bb89b5c1dd978d507284be78a04cb2,2017-05-23 22:28:36,141,199.98,122.88,SP,0
65284,40dc2ba6f322a17626aac6244332828c,2017-10-05 21:39:05,117,40.00,14.10,RJ,0
14121,0d5ce32fbaa548555dac4bee840d7705,2017-07-01 15:47:06,110,189.80,76.42,RN,0
36002,5aaba6681a3a926180ccd6ffaeaf6bd8,2017-12-25 23:45:35,102,179.90,45.34,SC,0
89700,865aee0fd5873d8a2974df861e2526ef,2018-01-07 20:49:50,100,179.90,45.34,SC,0
7956,09c636e254bb18eeef943452017249f4,2017-10-27 17:03:40,98,169.90,70.41,MT,0
57095,65a7c3111cd734466eb706d544e6fae4,2017-03-23 17:36:38,97,194.70,33.18,SP,0


In [17]:
print("Distribución por mes de compra")
display(df_features["purchase_month"].value_counts().sort_index())

print("Distribución por día de la semana: 0=lunes y 6=domingo")
display(df_features["purchase_dayofweek"].value_counts().sort_index())


Distribución por mes de compra


purchase_month
1      7817
2      8196
3      9549
4      9101
5     10294
6      9231
7     10028
8     10544
9      4150
10     4478
11     7288
12     5513
Name: count, dtype: int64

Distribución por día de la semana: 0=lunes y 6=domingo


purchase_dayofweek
0    15660
1    15448
2    15039
3    14278
4    13642
5    10511
6    11611
Name: count, dtype: int64

In [18]:
display(
    df_features.groupby(TARGET_COL)["promised_days"]
    .agg(["count", "mean", "median", "min", "max"])
)


,count,mean,median,min,max
pedido_retrasado,,,,,
0,89658,24.356086,24.0,3,156
1,6531,23.281886,23.0,3,70


La comparación anterior permite observar una asociación inicial entre el plazo prometido y la demora. No demuestra causalidad ni garantiza que la variable sea útil por sí sola; su aporte real debe medirse dentro de la validación temporal del modelo.


## 9. Conclusiones y entrega al modelado

- La tabla original tiene una fila por pedido.
- El target oficial se construye con una función compartida y compara días calendario.
- El período oficial evita incluir meses incompletos o poco representativos.
- Al predecir en la aprobación quedan 96.189 pedidos.
- Las variables se construyen sin usar información posterior.
- Los pesos, dimensiones y cuotas inválidas se mantienen como faltantes para que el pipeline los trate correctamente.
- La clase retrasada es minoritaria, por lo que el modelado debe mirar recall, precision, F1, ROC-AUC y especialmente PR-AUC.

Este notebook termina en la auditoría y preparación. La división temporal, comparación de modelos, calibración y elección del umbral se realizan en los módulos compartidos del proyecto.
